# nnU-Net baseline for the boneseg paper evaluation

Trains nnU-Net v2 (2D) once per held-out sample, on the same 5 labelled slices per training sample that boneseg and the random forest get, and predicts the held-out sample's test slices. The input is the folder written by `paper/nnunet_export.py`, uploaded as the private Kaggle dataset `liubone-nnunet`. Download `predictions.zip` from the output and score it with `python paper/nnunet_score.py --pred-dir predictions`.

Training uses nnU-Net's 250-epoch trainer to fit three folds into one Kaggle session; say so in the paper, since the default is 1000 epochs.

In [ ]:
%pip install -q nnunetv2
import os, json, shutil, subprocess, glob
from pathlib import Path
src = Path(glob.glob("/kaggle/input/**/Dataset501_LiuBone", recursive=True)[0])
work = Path("/kaggle/working/nnunet")
for d in ("raw", "preprocessed", "results"):
    (work / d).mkdir(parents=True, exist_ok=True)
os.environ.update(nnUNet_raw=str(work / "raw"), nnUNet_preprocessed=str(work / "preprocessed"), nnUNet_results=str(work / "results"))
shutil.copytree(src, work / "raw" / "Dataset501_LiuBone", dirs_exist_ok=True)
info = json.loads((work / "raw" / "Dataset501_LiuBone" / "test_cases.json").read_text())
print(info["fold_of_sample"])
def run(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
run("nnUNetv2_plan_and_preprocess -d 501 -c 2d --verify_dataset_integrity")
shutil.copy(work / "raw" / "Dataset501_LiuBone" / "splits_final.json", work / "preprocessed" / "Dataset501_LiuBone" / "splits_final.json")

In [ ]:
for sample, fold in info["fold_of_sample"].items():
    run(f"nnUNetv2_train 501 2d {fold} -tr nnUNetTrainer_250epochs")

In [ ]:
pred = Path("/kaggle/working/predictions"); pred.mkdir(exist_ok=True)
for sample, fold in info["fold_of_sample"].items():
    inp = Path(f"/kaggle/working/test_{sample}"); inp.mkdir(exist_ok=True)
    for case in info["test_cases"][sample]:
        shutil.copy(work / "raw" / "Dataset501_LiuBone" / "imagesTs" / f"{case}_0000.png", inp)
    run(f"nnUNetv2_predict -i {inp} -o {pred} -d 501 -c 2d -f {fold} -tr nnUNetTrainer_250epochs")
shutil.make_archive("/kaggle/working/predictions", "zip", pred)
print(len(list(pred.glob('*.png'))), "predictions")